# Pipeline of execution: 

* Run `resample.py`
* Run `features_volumetric.py`
* Run `features_radiomic.py`
* Run `features_displacement.py`

In [ ]:
import pandas as pd

INPUT_PATH = "/mnt/study-data/pgirardi/graphs/image_data_sMCI_pMCI.txt"
OUTPUT_PATH = "/mnt/study-data/pgirardi/graphs/image_data_sMCI_pMCI_extremos.csv"

df = pd.read_csv(INPUT_PATH)
df["MRI_DATE"] = pd.to_datetime(df["MRI_DATE"])

parts = []
for id_pt, g in df.groupby("ID_PT", sort=False):
    g_smci = g.loc[g["GROUP"] == "sMCI"].sort_values("MRI_DATE", ascending=True)
    early_smci = g_smci.head(3).copy()

    g_pmci = g.loc[g["GROUP"] == "pMCI"].sort_values("MRI_DATE", ascending=True)
    late_pmci = g_pmci.tail(3).copy()

    parts.extend([early_smci, late_pmci])

df_ext = pd.concat(parts, ignore_index=True)
df_ext.to_csv(OUTPUT_PATH, index=False)

print("Salvo:" , OUTPUT_PATH)
print("df_ext:", df_ext.shape, "linhas |", df_ext["ID_PT"].nunique(), "pacientes")
df_ext.head(10)

In [ ]:
import pandas as pd

# df_ext já carregado; se não:
# df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

# 1 linha por paciente (metadados do conjunto)
pt = (
    df_ext.sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
    .groupby("ID_PT", as_index=False)
    .agg(
        GROUP=("GROUP", "first"),
        SEX=("SEX", "first"),
        n_linhas=("ID_IMG", "size"),
    )
)

# conjuntos = pacientes com exatamente 3 linhas
pt = pt[pt["n_linhas"] == 3].copy()
pt["n_conjuntos"] = 1  # 1 conjunto por paciente válido

# --- totais ---
n_conjuntos_total = len(pt)
n_linhas_total = int(pt["n_linhas"].sum())
print("Conjuntos (pacientes com 3 linhas):", n_conjuntos_total)
print("Linhas totais:", n_linhas_total)
print("Checagem linhas/3:", n_linhas_total / 3)

# --- por GROUP ---
print("\nPor GROUP:")
print(pt["GROUP"].value_counts().sort_index())
# ou só contagem de conjuntos:
# print(pt.groupby("GROUP")["n_conjuntos"].sum())

# --- por SEX ---
print("\nPor SEX:")
print(pt["SEX"].value_counts().sort_index())

# --- GROUP × SEX ---
print("\nGROUP × SEX:")
print(pd.crosstab(pt["GROUP"], pt["SEX"], margins=True))

In [11]:
# Leitura radiomicos e volumetricos

import pandas as pd

ab = "abordagem_4_sMCI_pMCI_extremos"

vol_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_volumetric.csv"

rad_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_radiomic.csv"

vol_df = pd.read_csv(vol_path)
rad_df = pd.read_csv(rad_path)


In [12]:
# Normalizacao pelo ICV (mask_mm3 global)

# ICV = volume total da mascara do encefalo (linha __global__ em features_volumetric)
# Vamos anexar o ICV por ID_IMG e normalizar apenas as features dependentes de tamanho.

import numpy as np

# 1) ICV por ID_IMG
vol_df["ID_IMG"] = vol_df["ID_IMG"].astype(str).str.strip()

icv_df = (
    vol_df.loc[vol_df["roi"].astype(str) == "__global__", ["ID_IMG", "mask_mm3"]]
    .drop_duplicates(subset=["ID_IMG"], keep="last")
    .rename(columns={"mask_mm3": "ICV_mask_mm3"})
)

icv_df["ICV_mask_mm3"] = pd.to_numeric(icv_df["ICV_mask_mm3"], errors="coerce")

# 2) Merge com radiomicos
rad_df["ID_IMG"] = rad_df["ID_IMG"].astype(str).str.strip()
merged = rad_df.merge(icv_df, on="ID_IMG", how="left", validate="many_to_one")

missing_icv = int(merged["ICV_mask_mm3"].isna().sum())
if missing_icv:
    raise ValueError(
        f"ICV ausente para {missing_icv} linhas radiomicas. "
        "Verifique se todos os ID_IMG do radiomico existem no volumetrico (linha __global__)."
    )

# 3) Features a normalizar (dependentes de tamanho)
cols_to_norm = [
    # first-order
    "original_firstorder_Energy",
    "original_firstorder_TotalEnergy",
    # shape (absolutas)
    "original_shape_MeshVolume",
    "original_shape_VoxelVolume",
    "original_shape_SurfaceArea",
    "original_shape_LeastAxisLength",
    "original_shape_MajorAxisLength",
    "original_shape_MinorAxisLength",
    "original_shape_Maximum2DDiameterColumn",
    "original_shape_Maximum2DDiameterRow",
    "original_shape_Maximum2DDiameterSlice",
    "original_shape_Maximum3DDiameter",
]

# mantém só as que existem no arquivo (robustez)
cols_to_norm = [c for c in cols_to_norm if c in merged.columns]

# garante numérico
for c in cols_to_norm:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")

# 4) Normaliza pelo ICV
# Observacao: algumas features (p.ex. area) ficam com unidade relativa a volume;
# aqui seguimos a definicao pedida: dividir pelo volume total do encefalo.
merged[cols_to_norm] = merged[cols_to_norm].div(merged["ICV_mask_mm3"], axis=0)

# 5) CSV final: sem colunas "brutas" não-normalizadas adicionais (mantemos só a versão normalizada)
# (as colunas normalizadas sobrescrevem as originais, então basta salvar o merged)
out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"
merged.to_csv(out_path, index=False)

out_path


'/mnt/study-data/pgirardi/graphs/csvs/abordagem_4_sMCI_pMCI/radiomics_merge.csv'

In [13]:
# Leitura dados técnicos e inserção dos equipamentos na planilha de atributos radiomicos

import pandas as pd

tech_path = f"/mnt/study-data/pgirardi/graphs/csvs/adnimerged.csv"

radiomics_merged_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"

tech_df = pd.read_csv(tech_path)
radiomics_merged_df = pd.read_csv(radiomics_merged_path)

In [14]:
# Inserção de variáveis técnicas (por ID_IMG)

import numpy as np
import pandas as pd

# A planilha "merge" será o radiomics_merged_df enriquecido com informações técnicas/clínicas.
# (Se você preferir outro nome, basta trocar a variável aqui.)
merge = radiomics_merged_df.copy()

# 1) Merge por ID_IMG trazendo as colunas necessárias
cols_needed = [
    "ID_IMG",
    "ID_PT",
    "SEX",
    "DIAG",
    "MRI_DATE",
    "FIELD_STRENGTH",
    "SLICE_THICKNESS",
    "MANUFACTURER",
    "MFG_MODEL",
]

tech_sub = tech_df.loc[:, cols_needed].copy()
tech_sub["ID_IMG"] = tech_sub["ID_IMG"].astype(str).str.strip()
tech_sub = tech_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

merge["ID_IMG"] = merge["ID_IMG"].astype(str).str.strip()
merge = merge.merge(tech_sub, on="ID_IMG", how="left", validate="many_to_one")

# 2) Define batch (scanner) a partir das variáveis do equipamento
merge["batch"] = (
    merge[["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH", "SLICE_THICKNESS"]]
    .astype(str)
    .agg("_".join, axis=1)
)

missing_any = int(merge[cols_needed[1:]].isna().any(axis=1).sum())
print(f"Linhas sem alguma info técnica/clínica necessária (após merge): {missing_any}")



Linhas sem alguma info técnica/clínica necessária (após merge): 0


# Merge atributos unitários

In [15]:
import pandas as pd

rad_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"
disp_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_displacement_unitary.csv"
out_all_unit = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_features_unitary_merge.csv"

rad = pd.read_csv(rad_path)
print(rad.shape)
disp = pd.read_csv(disp_path)
print(disp.shape)

# (Opcional, mas recomendado) padroniza tipos das chaves
for df in (rad, disp):
    df["ID_IMG"] = df["ID_IMG"].astype(str).str.strip()
    df["roi"] = df["roi"].astype(str).str.strip()
    df["side"] = df["side"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()

all_unit = rad.merge(
    disp,
    on=["ID_IMG", "roi", "side", "label"],
    how="left",
    validate="one_to_one",  # se der erro aqui, há duplicatas em algum arquivo
)

all_unit.to_csv(out_all_unit, index=False)
print(all_unit.shape)

(48480, 112)
(48480, 56)
(48480, 164)


# Corrigir a sequencia desse pipeline
- remover as colunas que não fazem mais sentido
- remover as colunas label roi
- converter pra 0/1 as colunas SEX e GROUP

In [ ]:
# Catálogo de conjuntos: 3 imagens consecutivas por ID_PT (image_data_sMCI_pMCI_extremos.csv)
import pandas as pd

IMAGES_CSV = "/mnt/study-data/pgirardi/graphs/image_data_sMCI_pMCI_extremos.csv"
DAYS_PER_MONTH = 365.2425 / 12.0


def build_set_catalog(images_path: str = IMAGES_CSV) -> pd.DataFrame:
    """Por ID_PT, cada bloco de 3 linhas (MRI_DATE crescente) define i1, i2, i3."""
    img = pd.read_csv(images_path)
    img["ID_IMG"] = img["ID_IMG"].astype(str).str.strip()
    img["ID_PT"] = img["ID_PT"].astype(str).str.strip()
    img["MRI_DATE"] = pd.to_datetime(img["MRI_DATE"], errors="coerce")

    img = img.sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
    pos_in_pt = img.groupby("ID_PT", sort=False).cumcount()
    img["TRIPLET_IDX"] = (pos_in_pt // 3).astype(int)
    img["_pos"] = (pos_in_pt % 3).astype(int)
    img["COMBINATION_NUMBER"] = img["TRIPLET_IDX"] + 1

    complete_mask = img.groupby(["ID_PT", "TRIPLET_IDX"])["_pos"].transform("nunique") == 3
    n_drop = int((~complete_mask).sum())
    if n_drop:
        print(f"Aviso: descartando {n_drop} linhas em conjuntos incompletos (≠3 imagens).")
    img = img.loc[complete_mask].copy()

    idx_cols = ["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"]
    wide_img = (
        img.pivot_table(index=idx_cols, columns="_pos", values="ID_IMG", aggfunc="first")
        .rename(columns={0: "ID_IMG_i1", 1: "ID_IMG_i2", 2: "ID_IMG_i3"})
        .reset_index()
    )
    wide_date = (
        img.pivot_table(index=idx_cols, columns="_pos", values="MRI_DATE", aggfunc="first")
        .rename(columns={0: "MRI_DATE_i1", 1: "MRI_DATE_i2", 2: "MRI_DATE_i3"})
        .reset_index()
    )
    trip = wide_img.merge(wide_date, on=idx_cols, how="inner", validate="one_to_one")

    trip["t12"] = (
        (trip["MRI_DATE_i2"] - trip["MRI_DATE_i1"]).dt.total_seconds() / 86400.0
    ) / DAYS_PER_MONTH
    trip["t13"] = (
        (trip["MRI_DATE_i3"] - trip["MRI_DATE_i1"]).dt.total_seconds() / 86400.0
    ) / DAYS_PER_MONTH
    trip["t23"] = (
        (trip["MRI_DATE_i3"] - trip["MRI_DATE_i2"]).dt.total_seconds() / 86400.0
    ) / DAYS_PER_MONTH
    trip = trip.drop(columns=["MRI_DATE_i1", "MRI_DATE_i2", "MRI_DATE_i3"])

    cov_cols = [c for c in ["ID_IMG", "GROUP", "SEX", "DIAG", "AGE"] if c in img.columns]
    if cov_cols:
        cov = img[cov_cols].copy()
        cov["ID_IMG"] = cov["ID_IMG"].astype(str).str.strip()
        cov = cov.drop_duplicates(subset=["ID_IMG"], keep="last")
        trip = trip.merge(
            cov.rename(columns={"ID_IMG": "ID_IMG_i1"}),
            on="ID_IMG_i1",
            how="left",
            validate="many_to_one",
        )

    print(f"Conjuntos: {len(trip)} | Pacientes: {trip['ID_PT'].nunique()}")
    return trip


In [16]:
# Atributos unitários por conjunto (3 imagens × ROI) — formato long
#
# Entrada:
# - image_data_sMCI_pMCI_extremos.csv (3 linhas consecutivas por ID_PT = i1/i2/i3)
# - all_features_unitary_merge.csv (atributos unitários por ID_IMG × ROI)
#
# Saída:
# - all_unitary_features.csv (1 linha por conjunto × ROI × imagem do conjunto)

import os
import numpy as np
import pandas as pd

unit_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_features_unitary_merge.csv"
out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_unitary_features.csv"

trip = build_set_catalog(IMAGES_CSV)

long_imgs = trip.melt(
    id_vars=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "t12", "t13", "t23"],
    value_vars=["ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3"],
    var_name="pair",
    value_name="ID_IMG_ref",
)

_pair_map = {"ID_IMG_i1": 1, "ID_IMG_i2": 2, "ID_IMG_i3": 3}
long_imgs["pair"] = long_imgs["pair"].map(_pair_map).astype(int)
long_imgs["ID_IMG_ref"] = long_imgs["ID_IMG_ref"].astype(str).str.strip()

long_imgs = long_imgs.merge(
    trip[["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3"]],
    on=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"],
    how="left",
    validate="many_to_one",
)

unit = pd.read_csv(unit_path)
for c in ("ID_IMG", "roi", "side", "label"):
    unit[c] = unit[c].astype(str).str.strip()

roi_keys = ["ID_IMG", "roi", "side", "label"]
unit = unit.drop_duplicates(subset=roi_keys, keep="last")

_reserved = {
    "ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "pair",
    "ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3", "t12", "t13", "t23",
    "GROUP", "SEX", "DIAG", "AGE", "TIME_PROG",
    "FIELD_STRENGTH", "SLICE_THICKNESS", "MANUFACTURER", "MFG_MODEL",
    "MRI_DATE", "batch", "ID_IMG_ref",
}
_drop_from_unit = [c for c in unit.columns if c in _reserved and c not in roi_keys and c != "ID_IMG"]
if _drop_from_unit:
    unit = unit.drop(columns=_drop_from_unit)

out = long_imgs.merge(
    unit.rename(columns={"ID_IMG": "ID_IMG_ref"}),
    on=["ID_IMG_ref"],
    how="inner",
    validate="many_to_many",
)

img_meta = pd.read_csv(IMAGES_CSV)
img_meta["ID_IMG"] = img_meta["ID_IMG"].astype(str).str.strip()
img_cov_cols = [c for c in ["ID_IMG", "GROUP", "SEX", "DIAG", "AGE"] if c in img_meta.columns]
if img_cov_cols:
    img_cov = img_meta.loc[:, img_cov_cols].drop_duplicates(subset=["ID_IMG"], keep="last")
    out = out.merge(
        img_cov.rename(columns={"ID_IMG": "ID_IMG_ref"}),
        on="ID_IMG_ref",
        how="left",
        validate="many_to_one",
    )

tech_path = "/mnt/study-data/pgirardi/graphs/csvs/adnimerged.csv"
if os.path.exists(tech_path):
    tech_df = pd.read_csv(tech_path)
    tech_df["ID_IMG"] = tech_df["ID_IMG"].astype(str).str.strip()
    tech_keep = [
        c for c in [
            "ID_IMG", "GROUP", "SEX", "DIAG", "AGE", "TIME_PROG",
            "FIELD_STRENGTH", "SLICE_THICKNESS", "MANUFACTURER", "MFG_MODEL",
        ]
        if c in tech_df.columns
    ]
    tech_sub = tech_df.loc[:, tech_keep].drop_duplicates(subset=["ID_IMG"], keep="last")
    out = out.merge(
        tech_sub.rename(columns={"ID_IMG": "ID_IMG_ref"}),
        on="ID_IMG_ref",
        how="left",
        validate="many_to_one",
        suffixes=("", "_tech"),
    )

    for c in [
        "GROUP", "SEX", "DIAG", "AGE", "TIME_PROG",
        "FIELD_STRENGTH", "SLICE_THICKNESS", "MANUFACTURER", "MFG_MODEL",
    ]:
        c_tech = f"{c}_tech"
        if c_tech in out.columns:
            if c in out.columns:
                out[c] = out[c].where(out[c].notna(), out[c_tech])
            else:
                out[c] = out[c_tech]
            out = out.drop(columns=[c_tech])

tech_for_batch = [c for c in ["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH", "SLICE_THICKNESS"] if c in out.columns]
if tech_for_batch:
    out["batch"] = out[tech_for_batch].astype(str).agg("_".join, axis=1)
else:
    out["batch"] = np.nan

delta_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features.csv"
delta_cols = pd.read_csv(delta_path, nrows=0).columns.tolist() if os.path.exists(delta_path) else []

required_prefix = [
    "ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "pair",
    "ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3", "roi", "side", "label",
    "t12", "t13", "t23", "GROUP", "SEX", "DIAG", "AGE", "TIME_PROG",
    "ID_IMG_ref", "FIELD_STRENGTH", "SLICE_THICKNESS", "MANUFACTURER", "MFG_MODEL", "batch",
]
for c in required_prefix:
    if c not in out.columns:
        out[c] = np.nan

ordered = [c for c in delta_cols if c in out.columns] if delta_cols else []
if not ordered:
    ordered = [c for c in required_prefix if c in out.columns]
ordered += [c for c in out.columns if c not in ordered]

if "ref_tag" in ordered and "ID_IMG_i3" in ordered:
    ordered = [c for c in ordered if c != "ref_tag"]
    idx = ordered.index("ID_IMG_i3") + 1
    ordered.insert(idx, "ref_tag")

out = out.loc[:, ordered]

out.to_csv(out_path, index=False)
print("Salvo:", out_path)
print("shape:", out.shape)


/mnt/study-data/pgirardi/tmp/ipykernel_1023446/509875895.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_imgs["pair"] = long_imgs["pair"].replace({"ID_IMG_i1": 1, "ID_IMG_i2": 2, "ID_IMG_i3": 3}).astype(int)


Salvo: /mnt/study-data/pgirardi/graphs/csvs/abordagem_4_sMCI_pMCI/all_unitary_features.csv
shape: (76560, 178)
